In [ ]:
import pandas as pd
import numpy as np
from dash import Dash, html, dcc, callback, Output, Input
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

df1 = pd.read_csv('https://raw.githubusercontent.com/jerrengrimes/Projects/refs/heads/main/Solar%20Needs%20and%20Land%20Area/countries.csv');
df2 = pd.read_csv('https://raw.githubusercontent.com/jerrengrimes/Projects/refs/heads/main/Solar%20Needs%20and%20Land%20Area/owid-energy-data2.csv');
df3 = pd.read_csv('https://raw.githubusercontent.com/jerrengrimes/Projects/refs/heads/main/Solar%20Needs%20and%20Land%20Area/land_area.csv');
df4 = pd.read_csv('https://raw.githubusercontent.com/jerrengrimes/Projects/refs/heads/main/Solar%20Needs%20and%20Land%20Area/sunshine.csv')


df = pd.merge(df1, df2, on='name', how='outer')
df = pd.merge(df, df3, on='name', how='outer')
df = pd.merge(df, df4, on='name', how='outer')


df['energy_use (TWh/yr)'] = df['Energy use per capita (kWh)']*1E-9*(df['population'])
df['solar_panel_generation (W/m2)'] = 150
df['area_needed (km2)'] = (df['energy_use (TWh/yr)']/df['sunlight (hours/yr)'])*(10**12)/df['solar_panel_generation (W/m2)']*(10**-6)
df['percentage area (%)'] = df['area_needed (km2)']*100/df['sq km']
df = df.dropna(ignore_index=True)

lastupdate = "11-11-26"
zooml = 3

app = Dash()
fig = go.Figure(go.Scattermap())

def plot_country_with_square(country_name, center_lat, center_lon, area_km2):
    fig = go.Figure()
    fig.update_layout(margin={"r":0,"t":50,"l":0,"b":20})

    # Calculate half-side length in kilometers
    half_side_km = np.sqrt(area_km2) / 2.0
    
    # Earth's radius in kilometers
    R = 6378.1
    
    # Latitude delta (1 degree lat ~ 111 km everywhere)
    dlat = (half_side_km / R) * (180.0 / np.pi)
    
    # Longitude delta depends on latitude due to meridian convergence
    dlon = (half_side_km / (R * np.cos(np.radians(center_lat)))) * (180.0 / np.pi)
    
    # Define corners of the square (closed loop)
    square_lats = [
        center_lat + dlat,  # Top-Left
        center_lat + dlat,  # Top-Right
        center_lat - dlat,  # Bottom-Right
        center_lat - dlat,  # Bottom-Left
        center_lat + dlat   # Close loop
    ]
    
    square_lons = [
        center_lon - dlon,
        center_lon + dlon,
        center_lon + dlon,
        center_lon - dlon,
        center_lon - dlon
    ]


    fig.add_trace(
        go.Scattermap(
            lon=square_lons,
            lat=square_lats,
            mode="lines",  
            fill="toself",
            fillcolor="rgba(255, 0, 0, 0.6)",
            line=dict(color="red", width=0),
            showlegend=False,
        )
    )

    # Configure map display layout
    fig.update_layout(
        height=500,
        width=500,
        map=dict(
            style="carto-positron",
            center=dict(lat=center_lat, lon=center_lon),
            zoom=zooml,
        ),
    )

    return fig

banner = html.Header(
    className="banner",
    children=[
        # Title Container
        html.Div(
            children=[
                html.H1("Electricity Usage vs. Solar Availability", style={"margin": "0", "fontSize": "40px"}),
                html.P("How much land area converted to solar is necessary to meet the electricity usage demand?", style={"margin": "5px 0 0 0", "opacity": "0.9"}),
                html.Div(children=[
                    html.Button(f"Last Updated: {lastupdate}",style={"border": "1px","margin": "20px 20px 0 0", "opacity": "1", "background-color": "#ffe599","fontFamily": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif"}),
                    html.Button("Created by: Jerren Grimes, PhD",style={"border": "1px","margin": "20px 20px 0 0", "opacity": "1", "background-color": "#ffe599","fontFamily": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif"}),
                    ],)
                         ],
            style={"flexGrow": "1"}
        ),
    ],
    style={
        "backgroundColor": "#0641c8",  # Dark slate background
        "color": "#ffffff",             # White text
        "padding": "20px 30px",         # Inner spacing
        "display": "flex",              # Flexible alignment
        "alignItems": "center",         # Vertically center items
        "justifyContent": "space-between",
        "boxShadow": "0 2px 4px rgba(0,0,0,0.2)",
        "width": "100%",
        "fontFamily": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif",
        "boxSizing": "border-box"
    }
)


input1 = html.Div(
    #className="banner",
    children=[
        html.Div(
            children=[
                html.Label('Country', style={"margin": "0", "fontSize": "12px"}),
                dcc.Dropdown(df['name'].to_list(), 'United States',id='my-input'),],
                ),             
    ],
    style={
        #"backgroundColor": "#0641c8",  # Dark slate background
        #"color": "#ffffff",             # White text
        "padding": "10px 10px",         # Inner spacing
        #"display": "flex",              # Flexible alignment
        "alignItems": "center",         # Vertically center items
        "justifyContent": "space-between",
        "width": "30%",
        "fontFamily": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif",
        "boxSizing": "border-box"
    }
)

input2 = html.Div(
    #className="banner",
    children=[
        # Title Container
        html.Div(
            children=[
                html.Label('Solar Panel Generation (W/m2)', style={"margin": "0", "fontSize": "12px"}),
                dcc.Input(value='150', type='text',id='eff'),],
        ),              
    ],
    style={
        #"backgroundColor": "#0641c8",  # Dark slate background
        #"color": "#ffffff",             # White text
        "padding": "10px 10px",         # Inner spacing
        "display": "flex",              # Flexible alignment
        "alignItems": "center",         # Vertically center items
        "justifyContent": "space-between",
        "fontFamily": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif",
        "boxSizing": "border-box"
    }
)

        
app.layout = html.Div([
    
    html.Div(children=[
        banner,
        html.Br(),
        #html.Div(id='my-output',style={'color': 'blue', 'fontSize': 40}),

        html.Div(children =[
            input1,
            input2,
            ],
    style={
        'display': 'flex',
        'flexDirection': 'row',
        "alignItems": "left",
        'gap': '0px',
        #"flex": "1",
        #"minWidth": "0",
    }
),
        html.Br(),
        html.Div(children =[

            html.Div(
            #className="card",
                children=[
                    html.Div(
                        children=[
                            html.P('341785000',id='pop-output',style={'color': 'black', 'fontSize': 20}),
                            #html.P('Population (last updated:',style={'color': 'black', 'fontSize': 10}),
                            html.P('1',id='pop-year',style={'color': 'black', 'fontSize': 10})
                ],)
                
                        ],
                style={
                    "backgroundColor": "#ffffff",
                    "color": "#ffffff",
                    "padding": "20px 20px",
                    "display": "flex",
                    "alignItems": "center",
                    "justifyContent": "space-between",
                    "boxShadow": "2px 2px 4px rgba(0,0,0,0.1)",
                    "flex": "1",
                    "minWidth": "0",
                    "fontFamily": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif",
                    "boxSizing": "border-box"
                }
            ),
            html.Div(
            #className="card",
                children=[
                    html.Div(
                        children=[
                            html.P('25652',id='usage-output',style={'color': 'black', 'fontSize': 20}),
                            #html.P('Annual Usage (TWh)',style={'color': 'black', 'fontSize': 10})
                            html.P('1',id='usage-year',style={'color': 'black', 'fontSize': 10})
                ],)
                
                        ],
                style={
                    "backgroundColor": "#ffffff",
                    "color": "#ffffff",
                    "padding": "20px 20px",
                    "display": "flex",
                    "alignItems": "center",
                    "justifyContent": "space-between",
                    "boxShadow": "2px 2px 4px rgba(0,0,0,0.1)",
                    "flex": "1",
                    "minWidth": "0",
                    "fontFamily": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif",
                    "boxSizing": "border-box"
                }
            ),
            html.Div(
            #className="card",
                children=[
                    html.Div(
                        children=[
                            html.P('9147420',id='area-output',style={'color': 'black', 'fontSize': 20}),
                            html.P('Country Land Area (km²)',style={'color': 'black', 'fontSize': 10})
                ],)
                
                        ],
                style={
                    "backgroundColor": "#ffffff",
                    "color": "#ffffff",
                    "padding": "20px 20px",
                    "display": "flex",
                    "alignItems": "center",
                    "justifyContent": "space-between",
                    "boxShadow": "2px 2px 4px rgba(0,0,0,0.1)",
                    "flex": "1",
                    "minWidth": "0",
                    "fontFamily": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif",
                    "boxSizing": "border-box"
                }
            ),
            html.Div(
            #className="card",
                children=[
                    html.Div(
                        children=[
                            html.P('0.74',id='percent-output',style={'color': 'black', 'fontSize': 20}),
                            html.P('Area Needed for Solar Panels/Country Land Area (%)',style={'color': 'black', 'fontSize': 10})
                ],)
                
                        ],
                style={
                    "backgroundColor": "#ffffff",
                    "color": "#ffffff",
                    "padding": "20px 20px",
                    "display": "flex",
                    "alignItems": "center",
                    "justifyContent": "space-between",
                    "boxShadow": "2px 2px 4px rgba(0,0,0,0.1)",
                    "flex": "1",
                    "minWidth": "0",
                    "fontFamily": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif",
                    "boxSizing": "border-box"
                }
            ),
            html.Div(
            #className="card",
                children=[
                    html.Div(
                        children=[
                            html.P('2500.0',id='sunlight-output',style={'color': 'black', 'fontSize': 20}),
                            html.P('Sunlight in a year (hours)',style={'color': 'black', 'fontSize': 10})
                ],)
                
                        ],
                style={
                    "backgroundColor": "#ffffff",
                    "color": "#ffffff",
                    "padding": "20px 20px",
                    "display": "flex",
                    "alignItems": "center",
                    "justifyContent": "space-between",
                    "boxShadow": "2px 2px 4px rgba(0,0,0,0.1)",
                    "flex": "1",
                    "minWidth": "0",
                    "fontFamily": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif",
                    "boxSizing": "border-box"
                }
            )

            ],
    style={
        'display': 'flex',
        'flexDirection': 'row',
        'gap': '50px',
        'width': '100%',
    }
),
        html.Br(),
#from dash import html, dcc

html.Div(
    dcc.Graph(
        id="my-map", 
        figure=plot_country_with_square(country_name="France", center_lat=46.2276, center_lon=2.2137, area_km2=1000),
        style={
            'width': '40%',     # Graph takes up 40% of the screen width
            'height': '600px'   # Optional: enforce consistent height
        }
    ),
    style={
        'display': 'flex',
        'justifyContent': 'center',  # Centers the child horizontally
        'width': '100%'              # Parent container spans the full page width
    }
),
        

        #html.P([html.Br()], style={"margin": "5px 0 0 0", "opacity": "0.9"}),
        
        html.P(["So what exactly are we looking at in this map?\
        The red square represents the total land area that, if replaced with solar panels,\
        would match the electricity demand of that specific country taking into account panel\
        efficiency in the input box and estimated useful sunlight. "\
        ,html.Strong("This is just a thought exercise."),\
        " Of course it is completely impractical to place all of the solar panels in a single area of "\
        ,html.Em("any"),\
        " country. This also does not take into account the cost to install these panels or the materials that go into them.\
        And solar does not work at night so grid scale energy storage is needed. But isn't it interesting to see how little solar as a percentage of land area is needed to completely supply the energy needs for any country?"], 
        style={"margin": "20px 20px 0 0", "opacity": "1","fontFamily": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif"}),
        
        html.Br(),
        html.Br(),
        #html.P("I hope to update the database as I find better data sources.", style={"margin": "20px 20px 0 0", "opacity": "1","fontFamily": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif"}),
        
        html.Div(
            #className="card",
                children=[
                    html.Div(
                        children=[
                            html.P([html.Strong("Sources")],id='bibliography',style={'color': 'black', 'fontSize': 14}),
                            html.P("1. Country's population and energy usage: Hannah Ritchie, Pablo Rosado, and Max Roser (2023) - “Energy” Published online at OurWorldinData.org. Retrieved from: 'https://ourworldindata.org/energy' [Online Resource]",style={'color': 'black', 'fontSize': 10}),
                            html.P("2. Country's land area: https://data.worldbank.org/indicator/AG.LND.TOTL.K2",style={'color': 'black', 'fontSize': 10}),
                            html.P("3. Sunlight hours in a year: https://www.nomadino.com/en/leaderboard/climate-weather/sunshine-hours/",style={'color': 'black', 'fontSize': 10}),
                            html.P("4. Efficiency of solar panels at 150W: https://www.energydawnice.com/solar-panel-output-per-square-meter/",style={'color': 'black', 'fontSize': 10}),
                            html.P([html.Em("Sources will be updated as I find better ones.")],style={'color': 'black', 'fontSize': 10})
                    ,
                ],)
                
                        ],
                style={
                    "backgroundColor": "#ffffff",
                    "color": "#ffffff",
                    "padding": "20px 20px",
                    #"display": "flex",
                    "alignItems": "center",
                    "justifyContent": "space-between",
                    "boxShadow": "2px 2px 4px rgba(0,0,0,0.1)",
                    "flex": "1",
                    "minWidth": "0",
                    "fontFamily": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif",
                    "boxSizing": "border-box"
                }
            ),
        
    ], style={'padding': 10, 'flex': 1}),
    
], style={'display': 'flex', 'flexDirection': 'row'})


@callback(
    Output(component_id='pop-output', component_property='children'),
    Output(component_id='usage-output', component_property='children'),
    Output(component_id='area-output', component_property='children'),
    Output(component_id='percent-output', component_property='children'),
    Output(component_id='sunlight-output', component_property='children'),
    Output(component_id='pop-year', component_property='children'),
    Output(component_id='usage-year', component_property='children'),
    Output(component_id='my-map', component_property='figure'),
    Input(component_id='my-input', component_property='value'),
    Input(component_id='eff', component_property='value')
    
)

def update_output_div(input_value,eff):
    df['solar_panel_generation (W/m2)'] = int(eff)
    df['area_needed (km2)'] = (df['energy_use (TWh/yr)']/df['sunlight (hours/yr)'])*(10**12)/df['solar_panel_generation (W/m2)']*(10**-6)
    df['percentage area (%)'] = df['area_needed (km2)']*100/df['sq km']

    df2 = df[df['name'] == input_value]
    pop = f"{int((df2['population'].iloc[0])):,}"
    usage = f"{int((df2['energy_use (TWh/yr)'].iloc[0])):,}"
    area = f"{int((df2['sq km'].iloc[0])):,}"
    percent = f"{(df2['percentage area (%)'].iloc[0]):.2} %"
    sunlight = f"{int((df2['sunlight (hours/yr)'].iloc[0])):,}"
    popyear = f"{int(df2['energy year'].iloc[0])}"

    fig = plot_country_with_square(
        country_name=input_value,
        center_lat=df2['latitude'].iloc[0],
        center_lon=df2['longitude'].iloc[0],
        area_km2=df2['area_needed (km2)'].iloc[0]
    )

    return f'{pop}',f'{usage}',f'{area}',f'{percent}',f'{sunlight}',f'Population (last year updated: {popyear})',f'Energy Usage (TWh) (last year updated: {popyear})', fig


if __name__ == '__main__':
    app.run(debug=True)